# 00 — Why RAG Exists

Say you build **StoryVerse AI** — a chatbot that answers questions about movies, books, and stories. You ship it. The first user asks:

> "What happens at the end of *Kalki 2898 AD*?"

The bot answers right away. Sounds sure of itself. And it's completely wrong.

That's the bug we're chasing in this notebook. Not a small one — it's baked into how these models work. By the end, you'll see exactly why it happens, and why "just write a better prompt" won't fix it.


## It only knows what it studied

Here's the thing about the model behind StoryVerse AI: it studied once, a long time ago, and then stopped. Everything it read got squeezed into a giant pile of numbers inside it. After that, it never reads anything new. When you ask it a question, it isn't looking anything up — it's remembering.

Think of a student who crammed for six months, then walked into an exam with no notes, no phone, no internet. Whatever they didn't memorize, they simply don't know. And here's the tricky part: if a question *feels* close enough to something they studied, they'll still answer — confidently — even when they're wrong.

That's exactly what just happened with *Kalki 2898 AD*. The model never read anything about it, so it guessed and hoped for the best.

There's a technical name for this kind of memory: **parametric memory** — knowledge that got frozen into the model's parameters (its "weights") back when it was trained. It's the only kind of memory a raw LLM has on its own. Keep this term in your back pocket; we'll contrast it with a second kind of memory later in this notebook.

Let's poke at this from a few angles and see how deep the problem goes.


In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# openai/gpt-oss-20b is Groq's current free-tier replacement for the
# now-deprecated llama-3.1-8b-instant (deprecated June 2026).
MODEL = "openai/gpt-oss-20b"


def ask_llm(question: str) -> str:
    """Bare LLM call — no context, no tools, just whatever the model remembers."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=0.2,
    )
    return response.choices[0].message.content


## Angle 1 — it doesn't know anything new

The model's "studying" stopped on a fixed date — this is officially called the model's **training cutoff**. Anything that happened after that is a total blank. Let's ask about something recent and watch it either dodge the question or guess.


In [ ]:
print("LLM Response:", ask_llm("Who won the 2026 IPL final, and what was the final score?"))


Look closely at what came back: either a vague "I'm not sure" answer, or a made-up score said with total confidence. Either way, the model has no way to actually know — that event happened after its study cutoff, and it has no way to go check.


## Angle 2 — it doesn't know your stuff either

The model studied public internet text. It has never seen *your* PDFs, *your* notes, or a story you just wrote five minutes ago. Let's write a story it has definitely never seen, and ask about it anyway.


In [ ]:
secret_story = """
Arjun had walked past the same bookshop in Hyderabad's Abids market a
hundred times, but tonight the door was open a crack, spilling warm
yellow light onto the wet street. Inside, past shelves of yellowed
paperbacks, stood a second doorway that hadn't been there before --
a shimmering outline, like heat rising off summer asphalt. When Arjun
stepped through, the smell of old paper vanished, replaced by salt air
and the sound of waves. He was standing on a beach under two moons,
and behind him, the bookshop door was gone.
"""

# We are NOT giving the model this story yet -- just asking about it,
# the same way our first StoryVerse user asked about Kalki 2898 AD.
print("LLM Response:", ask_llm("What happens to Arjun in the bookshop story?"))


This story exists nowhere except the cell above. The model can't have read it. Yet it probably still gave you a confident-sounding answer. Which brings us to the real question: if it doesn't know, why doesn't it just say so?


## Angle 3 — when it doesn't know, it doesn't tell you

This is the core of the problem, and it explains both Kalki and Arjun. A model like this doesn't really "answer questions" — under the hood, it's just really good at guessing the next word that sounds right, over and over, until it has written a whole answer. Sounding right and being right are two different things.

To see this at its most obvious, let's ask about something that flat-out doesn't exist.


In [ ]:
print("LLM Response:", ask_llm("Explain the ending of the 1987 film 'The Crimson Algorithm'."))


There is no such film — I made it up. Yet you probably just got a full plot summary, maybe even character names and a twist ending. That's not a glitch. It's the model doing exactly what it was built to do: write the most convincing-sounding next sentence. Without something real to check itself against, convincing is all you get. This is called **hallucination**, and it's the same failure behind Kalki and Arjun — the model never says "I don't know," it just writes something plausible instead.


## Angle 4 — you can't just hand it everything

Okay — obvious fix: what if we just paste all our knowledge straight into the question, every time? Then it wouldn't need to guess.

There's a wall here too. Every model can only read so much text in one go — that's called its **context window**.

```
[ System Prompt | Your Documents | Conversation History | Answer ]
|←————————————————— context window (e.g. 128k tokens) ——————————→|
```

Rough rule of thumb: **1 token ≈ 4 letters** of English text. A whole novel is roughly 500,000 tokens. You can't fit "all your documents" in one prompt — and even on the rare occasion you technically could, it would be slow and expensive.


## The tempting shortcut #1: bigger and bigger prompts

Pasting one or two documents into the prompt works fine when you only have one or two documents. Let's see exactly where that stops being true.


In [ ]:
for num_docs in [1, 5, 20, 100, 500, 5000]:
    avg_doc_chars = 800  # about one short story summary
    total_chars = num_docs * avg_doc_chars
    est_tokens = total_chars // 4
    print(f"{num_docs:>5} docs -> {total_chars:>9,} chars -> ~{est_tokens:>8,} tokens")


At 1 document this is nothing. By 5,000 documents you've blown past what any prompt can hold — and that's before you even think about cost or speed. Pasting more and more text into the prompt is a patch, not a fix. It runs straight into the wall from Angle 4.


## The tempting shortcut #2: retrain the model on your data

Next idea: **fine-tuning** — the technical term for retraining the model on our own stories so it actually "knows" them. Sounds right, but it causes new problems:

- **Costs real money** — retraining needs serious computing power, every time
- **Takes hours or days** — not the few seconds it should take to add one new story
- **Goes stale immediately** — the moment you add a new story, the retrained model is out of date again
- **Bad at remembering exact facts** — retraining changes how a model *behaves* (its tone, its style), it doesn't reliably make it memorize precise details you can pull back out later

Think of it like this: fine-tuning is retraining the chef from scratch every single time the restaurant changes its menu. What you actually want is much simpler — just hand the chef a recipe card right before they start cooking. That's the idea we're building toward.


## Two kinds of memory

Everything above comes down to one question: where does the model's knowledge live, and how do you update it? This split has a name in the literature: **parametric vs. non-parametric memory**.

| | **Parametric memory**<br>(baked into the model, via fine-tuning) | **Non-parametric memory**<br>(looked up on demand — what we're building) |
|---|---|---|
| Where the knowledge lives | Inside the model itself | In a separate store of documents |
| Updating it | Slow and expensive — retrain the whole model | Fast and cheap — just edit a document |
| How fresh is it | Frozen at training time | As fresh as your documents |
| Can you check its sources | No — it's a black box | Yes — you can point at exactly which document it used |
| How often does it guess wrong | More often — it's working from memory | Less often — it's working from a document in front of it |

Parametric memory is what caused every failure so far in this notebook. Non-parametric memory is the whole idea behind RAG.


## So what actually fixes this?

The fix isn't a smarter model or a bigger prompt. It's giving the model a lookup step, right before it answers:

```
User Question
     |
     v
Search our own documents for anything relevant
     |
     v
Pull out the most relevant bits
     |
     v
[ System Prompt + Those Relevant Bits + Question ] --> Model --> Answer
```

This is called **RAG — Retrieval-Augmented Generation.** Nothing about the model changes. We just stop asking it to answer from memory alone, and start handing it the recipe card first. Feeding a model real documents like this is often called **grounding** it — the answer is now anchored ("grounded") in something real, instead of floating free on parametric memory alone.


## Where this leaves us

Four things broke StoryVerse AI, and they all trace back to one root cause — the model only knows what got baked into it during training, on a fixed day, and nothing after:

1. **Kalki 2898 AD** — happened after the model's cutoff, so it guessed
2. **Arjun's bookshop story** — never existed anywhere the model could have read it, so it guessed
3. **The Crimson Algorithm** — doesn't exist at all, so it guessed — and guessing confidently is what "hallucination" means
4. **Pasting everything in by hand** — runs into the context window wall, and doesn't scale anyway

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Training cutoff | The date after which the model knows nothing |
| Hallucination | Confidently making something up because it sounds plausible |
| Context window | The max amount of text the model can read in one go |
| Parametric memory | Knowledge frozen inside the model's weights |
| Non-parametric memory | Knowledge fetched from outside documents, on demand |
| Fine-tuning | Retraining the model on your own data |
| Grounding | Backing an answer with a real, checkable document |
| RAG (Retrieval-Augmented Generation) | Look something up first, then answer |

**Next notebook:** we try the simplest possible fix — manually pasting Arjun's story into the prompt ourselves — and see it actually work. Then we push on it until it breaks, so we know exactly why the rest of this series exists.

**Try it yourself:** ask your model a question about something that happened last week. Read the answer carefully — is it honestly saying "I don't know," or is it guessing and hoping you won't notice?
